# 模块概述

WtExecMon 是 WonderTrader 执行监控模块，负责管理执行器、交易通道、行情通道等组件的生命周期，协调各组件协同工作，实现智能订单执行。主要包括：
- 执行器运行器核心管理
- 交易通道和行情通道管理
- 执行器工厂和执行器管理
- 数据管理和K线数据管理
- 仓位管理和目标仓位提交
- C接口导出供外部调用

1. **运行器层**（WtExecRunner）：
   - 核心运行器，管理执行器、交易通道、行情通道等组件的生命周期
   - 实现 IParserStub 接口，接收解析器推送的行情数据
   - 实现 IExecuterStub 接口，为执行器提供基础信息查询功能
   - 协调各组件协同工作，实现智能订单执行

2. **适配器管理层**（TraderAdapterMgr + ParserAdapterMgr）：
   - **TraderAdapterMgr**：交易适配器管理器，管理多个交易通道
   - **ParserAdapterMgr**：解析器适配器管理器，管理多个行情通道
   - 支持多通道配置和动态加载

3. **执行器管理层**（WtExecuterFactory + WtExecuterMgr）：
   - **WtExecuterFactory**：执行器工厂，创建和管理执行器实例
   - **WtExecuterMgr**：执行器管理器，管理执行器的运行
   - 支持本地执行器、差分执行器、分布式执行器等类型

4. **数据管理层**（WtSimpDataMgr + WTSBaseDataMgr + WTSHotMgr）：
   - **WtSimpDataMgr**：简单数据管理器，管理行情数据和K线数据
   - **WTSBaseDataMgr**：基础数据管理器，管理商品、合约、交易时段等基础数据
   - **WTSHotMgr**：热点合约管理器，管理主力合约、次主力合约等

5. **工具支持层**（ActionPolicyMgr + WtExecPorter）：
   - **ActionPolicyMgr**：开平策略管理器，管理开仓和平仓策略
   - **WtExecPorter**：C接口导出，提供跨语言调用接口

6. **数据流程**：
   - 行情数据：解析器 → ParserAdapter → WtExecRunner → WtSimpDataMgr → 执行器
   - 交易指令：执行器 → TraderAdapter → 交易通道
   - 目标仓位：外部调用 → WtExecRunner → WtExecuterMgr → 执行器

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef runnerClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef adapterClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef executerClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef dataClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef utilClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef porterClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IParserStub["IParserStub<br/>解析器存根接口<br/>• 行情数据推送<br/>• 订单队列推送<br/>• 订单明细推送<br/>• 成交明细推送"]:::interfaceClass
        IExecuterStub["IExecuterStub<br/>执行器存根接口<br/>• 实时时间查询<br/>• 商品信息查询<br/>• 交易会话查询<br/>• 热点合约查询<br/>• 交易日期查询"]:::interfaceClass
        IDataManager["IDataManager<br/>数据管理器接口<br/>• Tick切片查询<br/>• K线切片查询<br/>• 最新Tick查询"]:::interfaceClass
        IDataReader["IDataReader<br/>数据读取器接口<br/>• 历史Tick读取<br/>• 历史K线读取"]:::interfaceClass
    end

    %% 运行器层
    subgraph Runner["运行器层 - 核心运行器"]
        direction TB
        WtExecRunner["WtExecRunner<br/>执行器运行器<br/>• 组件生命周期管理<br/>• 行情数据处理<br/>• 执行器支持<br/>• 仓位管理<br/>• 数据管理"]:::runnerClass
    end

    %% 适配器管理层
    subgraph Adapters["适配器管理层 - 通道管理"]
        direction TB
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器<br/>• 管理多个交易通道<br/>• 交易指令转发<br/>• 交易回报处理"]:::adapterClass
        ParserAdapterMgr["ParserAdapterMgr<br/>解析器适配器管理器<br/>• 管理多个行情通道<br/>• 行情数据转发<br/>• 数据过滤和标准化"]:::adapterClass
    end

    %% 执行器管理层
    subgraph Executers["执行器管理层 - 执行器管理"]
        direction TB
        WtExecuterFactory["WtExecuterFactory<br/>执行器工厂<br/>• 创建执行器实例<br/>• 加载执行器工厂<br/>• 管理执行器类型"]:::executerClass
        WtExecuterMgr["WtExecuterMgr<br/>执行器管理器<br/>• 管理执行器运行<br/>• 处理行情数据<br/>• 管理目标仓位"]:::executerClass
    end

    %% 数据管理层
    subgraph DataLayer["数据管理层 - 数据管理"]
        direction TB
        WtSimpDataMgr["WtSimpDataMgr<br/>简单数据管理器<br/>• 实时Tick缓存<br/>• K线数据缓存<br/>• 历史数据读取<br/>• 时间管理"]:::dataClass
        WTSBaseDataMgr["WTSBaseDataMgr<br/>基础数据管理器<br/>• 商品信息管理<br/>• 合约信息管理<br/>• 交易时段管理<br/>• 节假日管理"]:::dataClass
        WTSHotMgr["WTSHotMgr<br/>热点合约管理器<br/>• 主力合约管理<br/>• 次主力合约管理<br/>• 合约切换管理"]:::dataClass
    end

    %% 工具支持层
    subgraph Utils["工具支持层 - 辅助功能"]
        direction TB
        ActionPolicyMgr["ActionPolicyMgr<br/>开平策略管理器<br/>• 开仓策略管理<br/>• 平仓策略管理<br/>• 品种规则映射"]:::utilClass
        WtExecPorter["WtExecPorter<br/>C接口导出<br/>• 模块初始化<br/>• 配置加载<br/>• 运行控制<br/>• 仓位管理<br/>• 日志记录"]:::porterClass
    end

    %% 继承关系
    WtExecRunner -.->|"实现"| IParserStub
    WtExecRunner -.->|"实现"| IExecuterStub
    WtSimpDataMgr -.->|"实现"| IDataManager
    WtSimpDataMgr -.->|"实现"| IDataReaderSink

    %% 运行器组合关系
    WtExecRunner -->|"包含"| TraderAdapterMgr
    WtExecRunner -->|"包含"| ParserAdapterMgr
    WtExecRunner -->|"包含"| WtExecuterFactory
    WtExecRunner -->|"包含"| WtExecuterMgr
    WtExecRunner -->|"包含"| WtSimpDataMgr
    WtExecRunner -->|"包含"| WTSBaseDataMgr
    WtExecRunner -->|"包含"| WTSHotMgr
    WtExecRunner -->|"包含"| ActionPolicyMgr

    %% 数据流关系
    ParserAdapterMgr -->|"推送行情"| WtExecRunner
    WtExecRunner -->|"更新数据"| WtSimpDataMgr
    WtExecRunner -->|"转发行情"| WtExecuterMgr
    WtExecRunner -->|"查询信息"| WTSBaseDataMgr
    WtExecRunner -->|"查询热点"| WTSHotMgr

    %% 执行器关系
    WtExecuterFactory -->|"创建"| WtExecuterMgr
    WtExecuterMgr -->|"使用"| TraderAdapterMgr
    WtExecuterMgr -->|"使用"| WtSimpDataMgr

    %% 数据管理器关系
    WtSimpDataMgr -->|"使用"| IDataReader
    WtSimpDataMgr -->|"查询基础数据"| WTSBaseDataMgr
    WtSimpDataMgr -->|"查询热点"| WTSHotMgr

    %% C接口关系
    WtExecPorter -.->|"调用"| WtExecRunner

    %% 应用样式
    class WtExecRunner runnerClass
    class TraderAdapterMgr,ParserAdapterMgr adapterClass
    class WtExecuterFactory,WtExecuterMgr executerClass
    class WtSimpDataMgr,WTSBaseDataMgr,WTSHotMgr dataClass
    class ActionPolicyMgr utilClass
    class WtExecPorter porterClass
    class IParserStub,IExecuterStub,IDataManager,IDataReader interfaceClass
```

# 执行器模块C接口导出 WtExecPorter.h/cpp
定义了执行器模块的C语言接口，供外部程序（如Python、C#等）通过动态库调用。采用C接口设计，确保跨语言兼容性和动态库的稳定导出。

## 获取执行器运行器单例实例 getRunner
```cpp
/**
 * @brief 获取执行器运行器单例实例
 * 
 * 使用静态局部变量实现单例模式，确保整个模块只有一个WtExecRunner实例。
 * 
 * @return 返回WtExecRunner实例的引用
 * 
 * 单例模式说明：
 * - 第一次调用时创建实例，后续调用返回同一个实例
 * - 生命周期：实例在程序结束时自动销毁
 */
WtExecRunner& getRunner()
{
	static WtExecRunner runner;
	return runner;
}
```

## 初始化执行器模块 init_exec
```cpp
/**
 * @brief 初始化执行器模块
 * 
 * 初始化执行器模块的日志系统和运行环境。
 * 该函数使用静态标志确保只初始化一次，避免重复初始化。
 * 
 * @param logCfg 日志配置文件路径或配置内容字符串
 * @param isFile 是否为文件路径，true表示logCfg是文件路径，false表示logCfg是配置内容
 * 
 * 初始化流程：
 * 1. 检查是否已初始化，如果已初始化则直接返回
 * 2. 调用WtExecRunner::init()初始化执行器运行器
 * 3. 设置初始化标志为true
 */
void init_exec(WtString logCfg, bool isFile /*= true*/)
{
	static bool inited = false; // 静态初始化标志，确保只初始化一次
	if (inited)
		return;
	getRunner().init(logCfg); // 调用执行器运行器的初始化方法，初始化日志系统
	inited = true;
}
```

## 配置执行器模块 config_exec
```cpp
/**
 * @brief 配置执行器模块
 * 
 * 从配置文件加载执行器、交易通道、行情通道等配置信息。
 * 如果配置文件路径为空，则使用默认配置文件"cfgexec.json"。
 * 
 * @param cfgfile 配置文件路径或配置内容字符串，如果为空字符串则使用默认配置文件
 * @param isFile 是否为文件路径，true表示cfgfile是文件路径，false表示cfgfile是配置内容
 */
void config_exec(WtString cfgfile, bool isFile /*= true*/)
{
	if (strlen(cfgfile) == 0)
		getRunner().config("cfgexec.json");
	else
		getRunner().config(cfgfile);
}
```

## 运行执行器模块 run_exec
```cpp
/**
 * @brief 运行执行器模块
 * 启动执行器模块的运行，包括启动行情通道和交易通道。
 * 该函数会阻塞当前线程，直到模块停止运行。
 */
void run_exec()
{
	getRunner().run();
}
```

## 写入日志 write_log
```cpp
/**
 * @brief 写入日志
 * 
 * 记录日志信息，支持不同日志级别和日志分类。
 * 如果指定了分类名称，则使用分类日志记录；否则使用默认日志记录。
 * 
 * @param level 日志级别，使用WTSLogLevel枚举值（如LL_DEBUG、LL_INFO、LL_WARN、LL_ERROR）
 * @param message 日志消息内容
 * @param catName 日志分类名称，如果为空字符串则使用默认分类
 */
void write_log(unsigned int level, WtString message, WtString catName)
{
	if (strlen(catName) > 0) // 如果指定了日志分类名称
	{
		WTSLogger::log_raw_by_cat(catName, (WTSLogLevel)level, message); // 按分类记录日志
	}
	else // 如果未指定日志分类名称
	{
		WTSLogger::log_raw((WTSLogLevel)level, message); // 记录到默认分类
	}
}
```

## 获取执行器模块版本信息 get_version
```cpp
/**
 * @brief 获取执行器模块版本信息
 * 
 * 返回执行器模块的版本信息字符串，包括平台类型、版本号、编译日期和时间。
 * 使用静态变量缓存版本字符串，避免重复构建。
 * 
 * @return 返回版本信息字符串指针，格式如："X64 1.0.0 Build@Mar 30 2020 12:00:00"
 * 
 * 版本信息格式：
 * - 平台类型：X64（64位Windows）、X86（32位Windows）、UNIX（Linux/Unix）
 * - 版本号：从WT_VERSION宏获取
 * - 编译日期：从__DATE__宏获取（格式：MMM DD YYYY）
 * - 编译时间：从__TIME__宏获取（格式：HH:MM:SS）
 * 
 * 注意事项：
 * - 返回的字符串指针指向静态存储区，不需要释放
 * - 多次调用返回相同的字符串指针
 * - 版本字符串在第一次调用时构建，后续调用直接返回缓存值
 */
WtString get_version()
{
	static std::string _ver;  // 静态版本字符串，缓存版本信息
	if (_ver.empty())  // 如果版本字符串为空，构建版本信息
	{
		_ver = PLATFORM_NAME;  // 添加平台名称（X64/X86/UNIX）
		_ver += " ";  // 添加空格分隔符
		_ver += WT_VERSION;  // 添加版本号（从WT_VERSION宏获取）
		_ver += " Build@";  // 添加构建标识
		_ver += __DATE__;  // 添加编译日期（从__DATE__宏获取，格式：MMM DD YYYY）
		_ver += " ";  // 添加空格分隔符
		_ver += __TIME__;  // 添加编译时间（从__TIME__宏获取，格式：HH:MM:SS）
	}
	return _ver.c_str();  // 返回版本字符串的C字符串指针
}
```

## 释放执行器模块资源 release_exec
```cpp
/**
 * @brief 释放执行器模块资源
 * 清理执行器模块占用的资源，停止日志系统。
 * 调用此函数后，模块将无法继续使用，需要重新初始化。
 */
void release_exec()
{
	getRunner().release();
}
```

## 设置目标仓位 set_position
```cpp
/**
 * @brief 设置目标仓位
 * 
 * 设置指定合约的目标持仓数量。
 * 
 * @param stdCode 标准合约代码，如"SHFE.rb2305"、"CFFEX.IF2303"等
 * @param targetPos 目标持仓数量，正数表示多头，负数表示空头，0表示平仓
 */
void set_position(WtString stdCode, double targetPos)
{
	getRunner().setPosition(stdCode, targetPos);
}
```

## 提交目标仓位 commit_positions
```cpp
/**
 * @brief 提交目标仓位
 * 将所有已设置的目标仓位提交给执行器执行。
 * 执行器会根据当前持仓和目标持仓的差异，生成相应的交易指令。
 */
void commit_positions()
{
	getRunner().commitPositions();
}
```

# 执行器运行器 WtExecRunner.h/cpp
```cpp
class WtExecRunner : public IParserStub, public IExecuterStub
```
WonderTrader执行器模块的核心运行器。负责管理执行器、交易通道、行情通道等组件的生命周期，协调各组件协同工作。

## 成员
- **组件管理器**
  - `TraderAdapterMgr _traders`：交易适配器管理器，管理多个交易通道
  - `ParserAdapterMgr _parsers`：解析器适配器管理器，管理多个行情通道
  - `WtExecuterFactory _exe_factory`：执行器工厂，创建和管理执行器实例
  - `WtExecuterMgr _exe_mgr`：执行器管理器，管理执行器的运行
- **配置与数据管理**
  - `WTSVariant* _config`：配置对象指针，存储加载的配置信息
  - `WtSimpDataMgr _data_mgr`：简单数据管理器，管理行情数据和K线数据
  - `WTSBaseDataMgr _bd_mgr`：基础数据管理器，管理商品、合约、交易时段等基础数据
  - `WTSHotMgr _hot_mgr`：热点合约管理器，管理主力合约、次主力合约等
  - `ActionPolicyMgr _act_policy`：开平策略管理器，管理开仓和平仓策略
- **仓位管理**
  - `wt_hashmap<std::string, double> _positions`：目标仓位映射表，键为合约代码，值为目标持仓数量

## 属性方法

### 初始化执行器运行器 init

### 配置执行器运行器 config

### 运行执行器运行器 run
```cpp
/**
 * @brief 运行执行器运行器

 * 运行流程：
 * 1. 启动解析器适配器管理器（行情通道），接收实时行情数据
 * 2. 启动交易适配器管理器（交易通道），处理交易指令
 * 3. 执行器根据行情数据和目标仓位执行交易逻辑
 * 
 * 异常处理：
 * - 使用try-catch捕获所有异常
 * - 捕获异常后打印堆栈跟踪信息到日志
 * - 确保程序不会因异常而崩溃
 */
void WtExecRunner::run()
{
	try
	{
		_parsers.run(); // 启动解析器适配器管理器（行情通道）
		_traders.run(); // 启动交易适配器管理器（交易通道）
	}
	catch (...)
	{
		print_stack_trace([](const char* message) { // 打印堆栈跟踪信息，使用lambda表达式作为回调函数
			WTSLogger::error(message); // 将堆栈跟踪信息记录到日志
		});
	}
}
```

### 释放执行器运行器资源 release
```cpp
/**
 * @brief 释放执行器运行器资源
 * 清理执行器运行器占用的资源，停止日志系统。
 * 调用此函数后，模块将无法继续使用，需要重新初始化。
 */
void WtExecRunner::release()
{
	WTSLogger::stop();
}
```

## 仓位管理

### 设置目标仓位 setPosition
```cpp
/**
 * @brief 设置目标仓位
 * 
 * 设置指定合约的目标持仓数量。
 * 目标仓位会被缓存，直到调用commitPositions()提交执行。
 * 
 * @param stdCode 标准合约代码，如"SHFE.rb2305"、"CFFEX.IF2303"等
 * @param targetPos 目标持仓数量，正数表示多头，负数表示空头，0表示平仓
 */
void WtExecRunner::setPosition(const char* stdCode, double targetPos)
{
	_positions[stdCode] = targetPos; // 将目标仓位存储到映射表中
}
```

### 提交目标仓位 commitPositions
```cpp
/**
 * @brief 提交目标仓位
 * 
 * 将所有已设置的目标仓位提交给执行器执行。
 * 执行器会根据当前持仓和目标持仓的差异，生成相应的交易指令。
 * 
 * 执行流程：
 * 1. 将目标仓位传递给执行器管理器
 * 2. 执行器管理器根据目标仓位生成交易指令
 * 3. 清空目标仓位缓存
 */
void WtExecRunner::commitPositions()
{
	_exe_mgr.set_positions(_positions);  // 将目标仓位传递给执行器管理器
	_positions.clear();  // 清空目标仓位缓存
}
```

## 组件管理

### 添加执行器工厂目录 addExeFactories
```cpp
/**
 * @brief 添加执行器工厂目录
 * 
 * 从指定目录加载执行器工厂动态库。
 * 
 * @param folder 执行器工厂目录路径
 * @return 加载成功返回true，失败返回false
 */
bool WtExecRunner::addExeFactories(const char* folder)
{
	return _exe_factory.loadFactories(folder);  // 从指定目录加载执行器工厂动态库
}
```

### 获取基础数据管理器 get_bd_mgr
```cpp
/**
 * @brief 获取基础数据管理器
 * 
 * 返回基础数据管理器的指针，用于访问商品、合约、交易时段等基础数据。
 * 
 * @return 返回基础数据管理器指针
 */
IBaseDataMgr*	get_bd_mgr() { return &_bd_mgr; }
```

### 获取热点合约管理器 get_hot_mgr
```cpp
/**
 * @brief 获取热点合约管理器
 * 
 * 返回热点合约管理器的指针，用于查询主力合约、次主力合约等。
 * 
 * @return 返回热点合约管理器指针
 */
IHotMgr* get_hot_mgr() { return &_hot_mgr; }
```

### 获取交易会话信息 get_session_info
```cpp
/**
 * @brief 获取交易会话信息
 * 
 * 根据会话ID或合约代码获取交易会话信息。
 * 
 * @param sid 会话ID或合约代码
 * @param isCode 是否为合约代码，true表示sid是合约代码，false表示sid是会话ID
 * @return 返回交易会话信息指针，未找到返回NULL
 */
WTSSessionInfo* WtExecRunner::get_session_info(const char* sid, bool isCode /* = true */)
{
	if (!isCode)  // 如果sid不是合约代码，而是会话ID
		return _bd_mgr.getSession(sid);  // 直接从基础数据管理器查询会话信息

	CodeHelper::CodeInfo codeInfo = CodeHelper::extractStdCode(sid, NULL);  // 从标准合约代码中提取交易所和品种代码
	WTSCommodityInfo* cInfo = _bd_mgr.getCommodity(codeInfo._exchg, codeInfo._product);  // 查询商品信息
	if (cInfo == NULL)  // 如果商品信息不存在
		return NULL;  // 返回NULL

	return cInfo->getSessionInfo();  // 返回商品信息中的交易会话信息
}
```

## IParserStub接口实现

### 处理实时主推行情 handle_push_quote
当解析器收到新的行情数据时调用
- 根据 quote 设置 WtHelper 的全局时间和全局交易日
- 数据管理器进行实时行情推送 `_data_mgr.handle_push_quote`
- 执行器管理器处理Tick数据 `_exe_mgr.handle_tick`
```cpp
/**
 * @brief 处理实时主推行情（IParserStub接口实现）
 * @param quote 最新的tick数据指针
 */
void WtExecRunner::handle_push_quote(WTSTickData* quote)
```

## IExecuterStub接口实现

### 获取实时时间 get_real_time
```cpp
/**
 * @brief 获取实时时间（IExecuterStub接口实现）
 * 
 * 返回当前实时时间戳（纳秒级）。
 * 
 * @return 返回当前实时时间戳（纳秒级）
 */
uint64_t WtExecRunner::get_real_time()
{
    // 组合日期和时间生成时间戳
	return TimeUtils::makeTime(_data_mgr.get_date(), _data_mgr.get_raw_time() * 100000 + _data_mgr.get_secs());
}
```

### 获取商品信息 get_comm_info
```cpp
/**
 * @brief 获取商品信息（IExecuterStub接口实现）
 * 
 * 根据标准合约代码获取对应的商品信息。
 * 
 * @param stdCode 标准合约代码
 * @return 返回商品信息对象指针，未找到返回NULL
 */
WTSCommodityInfo* WtExecRunner::get_comm_info(const char* stdCode)
{
	CodeHelper::CodeInfo codeInfo = CodeHelper::extractStdCode(stdCode, NULL); // 从标准合约代码中提取交易所和品种代码
	return _bd_mgr.getCommodity(codeInfo._exchg, codeInfo._product); // 从基础数据管理器查询商品信息
}
```

### 获取交易会话信息 get_sess_info
```cpp
/**
 * @brief 获取交易会话信息（IExecuterStub接口实现）
 * 
 * 根据标准合约代码获取对应的交易会话信息。
 * 
 * @param stdCode 标准合约代码
 * @return 返回交易会话信息对象指针，未找到返回NULL
 */
WTSSessionInfo* WtExecRunner::get_sess_info(const char* stdCode)
{
	CodeHelper::CodeInfo codeInfo = CodeHelper::extractStdCode(stdCode, NULL); // 从标准合约代码中提取交易所和品种代码
	WTSCommodityInfo* cInfo = _bd_mgr.getCommodity(codeInfo._exchg, codeInfo._product); // 查询商品信息
	if (cInfo == NULL)
		return NULL;

	return cInfo->getSessionInfo(); // 返回商品信息中的交易会话信息
}
```

### 获取热点合约管理器 get_hot_mon
```cpp
/**
 * @brief 获取热点合约管理器（IExecuterStub接口实现）
 * 
 * 返回热点合约管理器的指针。
 * 
 * @return 返回热点合约管理器指针
 */
virtual IHotMgr* get_hot_mon() override { return &_hot_mgr; }
```

### 获取交易日期 get_trading_day
```cpp
/**
 * @brief 获取交易日期（IExecuterStub接口实现）
 * 
 * 返回当前交易日期。
 * 
 * @return 返回当前交易日期（格式：YYYYMMDD）
 */
uint32_t WtExecRunner::get_trading_day()
{
	return _data_mgr.get_trading_day();
}
```

## 私有初始化方法

### 初始化交易通道 initTraders

### 初始化行情通道 initParsers

### 初始化执行器 initExecuters

### 初始化数据管理器 initDataMgr

### 初始化开平策略 initActionPolicy

# 简单数据管理器 WtSimpDataMgr.h/cpp
```cpp
class WtSimpDataMgr : public IDataReaderSink, public IDataManager
```
是 WonderTrader 执行器模块中的数据管理器，负责管理行情数据和K线数据的读取、缓存和查询。同时实现 IDataReaderSink 和 IDataManager 接口，作为数据读取器的接收者和数据管理器。

## 成员
- **核心组件指针**
  - `IDataReader* _reader`：数据读取器指针，用于读取历史Tick和K线数据
  - `WtExecRunner* _runner`：执行器运行器指针，用于获取基础数据管理器和热点合约管理器
  - `WTSSessionInfo* _s_info`：交易会话信息指针，用于时间转换和交易时段判断

- **数据缓存**
  - `DataCacheMap* _bars_cache`：K线缓存，缓存合成后的K线数据，键为"合约代码-周期-倍数"，值为K线数据对象
    - typedef `WTSHashMap`\<std::string\> DataCacheMap：定义数据缓存映射表类型，键为字符串，值为数据对象指针
  - `DataCacheMap* _rt_tick_map`：实时tick缓存，缓存实时Tick数据，键为合约代码，值为Tick数据对象
    - typedef `WTSHashMap`\<std::string\> DataCacheMap：定义数据缓存映射表类型，键为字符串，值为数据对象指针

- **时间信息**
  - `uint32_t _cur_date`：当前日期，格式如yyyyMMdd（YYYYMMDD）
  - `uint32_t _cur_act_time`：当前完整时间，格式如hhmmssmmm（HHMMSSmmm，毫秒级）
  - `uint32_t _cur_raw_time`：当前真实分钟，格式如hhmm（HHMM，不包括秒和毫秒）
  - `uint32_t _cur_min_time`：当前1分钟线时间，格式如hhmm（HHMM，1分钟K线的时间）
  - `uint32_t _cur_secs`：当前秒数，格式如ssmmm（SSmmm，包括秒和毫秒）
  - `uint32_t _cur_tdate`：当前交易日，格式如yyyyMMdd（YYYYMMDD）

## 核心属性

### 初始化数据管理器 init
```cpp
/**
 * @brief 初始化数据管理器
 * 
 * 初始化数据管理器，加载数据存储模块。
 * 
 * @param cfg 数据管理器配置对象
 * @param runner 执行器运行器指针，用于获取基础数据管理器和热点合约管理器
 * @return 初始化成功返回true，失败返回false
 */
bool WtSimpDataMgr::init(WTSVariant* cfg, WtExecRunner* runner)
{
	_runner = runner;
	return initStore(cfg->get("store"));
}
```

### 处理实时行情推送 handle_push_quote
该函数是数据管理器处理**实时行情推送**的核心入口。它负责接收最新的 Tick 数据，将其存入缓存，并根据 Tick 的时间戳更新系统内部维护的各类时间状态（如当前日期、当前秒数、当前 K 线时间等），以保证执行器的时间概念与最新市场行情保持同步。
* **有效性检查**
  * 首先判断传入的 `curTick` 指针是否为空，若为空则直接返回。
* **实时 Tick 缓存管理**
  * 检查 `_rt_tick_map`（实时 Tick 缓存表）是否存在，若不存在则创建。
  * 将最新的 Tick 数据存入缓存 `_rt_tick_map->add`。参数 `true` 表示如果该合约已存在缓存，则覆盖更新。
* **时间戳过滤（防止时间倒流）**
  * 获取 Tick 的日期 `uDate` 和时间 `uTime`。
  * **判断逻辑**：如果系统当前记录的日期 `_cur_date` 不为 0，且新 Tick 的日期小于当前日期，或者日期相同但时间小于当前已记录的时间 `_cur_act_time`。
  * **动作**：视为过期或乱序数据，直接返回，不更新系统时间状态。
* **更新基础时间状态**
  * `_cur_date`更新为 `uDate`。
  * `_cur_act_time`更新为 `uTime`。
* **计算衍生时间（分钟线时间处理）**
  * **提取时间分量**：
    * `_cur_raw_time` (HHMM) = 完整时间 `_cur_act_time` / 100000（去掉秒和毫秒）。
    * `_cur_secs` (SSmmm) = 完整时间 `_cur_act_time` % 100000（保留秒和毫秒）。
  * **转换为分钟数**：调用 `_s_info->timeToMinutes` 将 HHMM 转换为从交易日开始累计的分钟数。
  * **小节结束修正**：
    * 调用 `_s_info->isLastOfSection` 判断当前时间是否为某个交易小节的结束点（如 11:30）。
    * 如果是结束点，分钟数 `minutes` 减 1（归入上一分钟）。
  * **K 线时间偏移**：
    * 将 `minutes` 加 1。这是因为 WonderTrader 中 1 分钟 K 线的时间标签通常指该分钟结束的时间（例如 09:30:05 的 Tick 归属于 09:31 的 K 线）。
  * **生成分钟线时间**：
    * 调用 `_s_info->minuteToTime` 将修正后的分钟数转回 HHMM 格式，存入 `_cur_min_time`。
* **更新交易日**
  * 最后更新 `_cur_tdate` 为 Tick 数据中的交易日 `tradingdate`。

```cpp
/**
 * @brief 处理实时行情推送
 * 
 * 接收并处理实时行情数据，更新缓存和时间信息。
 * 
 * @param stdCode 标准合约代码
 * @param curTick 新的Tick数据指针
 */
void WtSimpDataMgr::handle_push_quote(const char* stdCode, WTSTickData* curTick)
```

## 数据查询工具方法

### 获取数据读取器指针 reader
```cpp
/**
 * @brief 获取数据读取器指针
 * 
 * 返回数据读取器的指针。
 * 
 * @return 返回数据读取器指针
 */
inline IDataReader* reader() { return _reader; }
```

### 获取当前原始时间 get_raw_time
```cpp
/**
 * @brief 获取当前原始时间
 * 
 * 返回当前真实分钟时间（不包括秒和毫秒）。
 * 
 * @return 返回当前真实分钟时间（格式：HHMM）
 */
inline uint32_t	get_raw_time() const { return _cur_raw_time; }
```

### 获取当前交易日 get_trading_day
```cpp
/**
 * @brief 获取当前交易日
 * 
 * 返回当前交易日。
 * 
 * @return 返回当前交易日（格式：YYYYMMDD）
 */
inline uint32_t	get_trading_day() const { return _cur_tdate; }
```

## IDataManager 接口实现

### 获取Tick数据切片 get_tick_slice
```cpp
/**
 * @brief 获取Tick数据切片（IDataManager接口实现）
 * 
 * 获取指定合约的Tick数据切片。
 * 
 * @param code 合约代码
 * @param count 数据条数
 * @param etime 截止时间戳，默认为0（当前时间）
 * @return 返回Tick数据切片指针，未找到返回NULL
 */
WTSTickSlice* WtSimpDataMgr::get_tick_slice(const char* code, uint32_t count, uint64_t etime /*= 0*/)
{
	if (_reader == NULL)
		return NULL;
	return _reader->readTickSlice(code, count, etime);
}
```

### 获取K线数据切片 get_kline_slice
该函数用于获取指定合约的历史 **K 线数据切片**。它不仅支持读取基础周期（如 1 分钟、1 天）的数据，还内置了 **K 线重采样/合成** 机制。当请求的 K 线周期倍数（`times`）大于 1 时（例如请求 5 分钟 K 线，但底层只有 1 分钟数据），它会自动读取基础数据并合成目标周期的 K 线，同时利用缓存机制提高后续查询效率。

* **读取器检查**
  * 如果底层数据读取接口 `_reader` 为空，直接返回 NULL。
* **模式一：基础周期读取（times == 1）**
  * 如果请求的周期倍数 `times` 为 1（即不需要合成）：
  * 直接调用底层 `_reader->readKlineSlice` 读取并返回结果。
* **模式二：复合周期读取与合成（times > 1）**
  * **生成缓存键**：
    * 获取合约的会话信息 `sInfo`。
    * 生成 Key 字符串，格式为 `"合约代码-周期枚举值-倍数"`（例如 `rb2010-m1-5`）。
  * **查询缓存**：
    * 初始化缓存容器 `_bars_cache`（如果为空）。
    * 尝试从缓存中获取对应的 `kData`。
  * **数据合成（缓存未命中或数据不足）**：
    * 如果缓存为空，或者缓存中的 K 线数量 `kData->size()` 小于请求数量 `count`：
      * **计算需求量**：`realCount = count * times + times`。为了合成足够的 N 根目标 K 线，需要读取更多的基础 K 线（乘以倍数并留有余量）。
      * **读取基础数据**：调用 `_reader->readKlineSlice` 读取 `realCount` 数量的基础周期 K 线 `rawData`。
      * **执行合成**：
        * 若读取成功，调用全局工厂 `g_dataFact.extractKlineData`，利用 `rawData`、`sInfo` 进行重采样，生成目标倍数的 K 线数据 `kData`。
        * 释放基础数据 `rawData`。
      * **更新缓存**：将合成好的 `kData` 存入 `_bars_cache`。
  * **构建切片返回**：
    * **计算返回数量**：`rtCnt` = min(缓存总数, 请求数量)。
    * **定位起始点**：`sIdx` = 缓存总数 - `rtCnt`（从后向前截取）。
    * **获取数据指针**：取出起始位置的 Bar 指针 `rtHead`。
    * **创建切片**：使用 `WTSKlineSlice::create` 基于 `rtHead` 和 `rtCnt` 创建切片对象并返回。

```cpp
/**
 * @brief 获取K线数据切片（IDataManager接口实现）
 * 
 * 获取指定合约的K线数据切片，支持不同周期和倍数。
 * 
 * @param stdCode 合约代码
 * @param period K线周期（如PERIOD_M1、PERIOD_M5等）
 * @param times 周期倍数，1表示基础周期，大于1表示合成周期
 * @param count 数据条数
 * @param etime 截止时间戳，默认为0（当前时间）
 * @return 返回K线数据切片指针，未找到返回NULL
 */
WTSKlineSlice* WtSimpDataMgr::get_kline_slice(const char* stdCode, WTSKlinePeriod period, uint32_t times, uint32_t count, uint64_t etime /*= 0*/)
```

### 获取最新Tick数据 grab_last_tick
```cpp
/**
 * @brief 获取最新Tick数据（IDataManager接口实现）
 * 
 * 获取指定合约的最新Tick数据。
 * 
 * @param code 合约代码
 * @return 返回最新Tick数据指针，未找到返回NULL
 * 
 * 注意事项：
 * - 返回的Tick数据需要调用者负责释放（调用retain/release）
 * - 如果实时Tick缓存不存在，返回NULL
 */
WTSTickData* WtSimpDataMgr::grab_last_tick(const char* code)
{
	if (_rt_tick_map == NULL)
		return NULL;

	WTSTickData* curTick = (WTSTickData*)_rt_tick_map->get(code); // 从缓存中获取指定合约的最新Tick数据
	if (curTick == NULL)
		return NULL;

	curTick->retain(); // 增加引用计数，确保数据不会被释放
	return curTick;
}
```

## IDataStoreListener 接口实现

### K线数据更新回调 on_bar
```cpp
/**
 * @brief K线数据更新回调（IDataStoreListener接口实现）
 * 
 * 当数据存储器更新K线数据时调用。
 * 
 * @param code 合约代码
 * @param period K线周期
 * @param newBar 新的K线数据指针
 * 
 * 当前实现：空实现，不处理K线更新通知
 */
void WtSimpDataMgr::on_bar(const char* code, WTSKlinePeriod period, WTSBarStruct* newBar)
{

}
```

### 所有K线数据更新完成回调 on_all_bar_updated
```cpp
/**
 * @brief K线数据更新完成回调（IDataStoreListener接口实现）
 * 
 * 当数据存储器完成所有K线数据更新时调用。
 * 
 * @param updateTime 更新时间戳
 * 
 * 当前实现：空实现，不处理更新完成通知
 */
void WtSimpDataMgr::on_all_bar_updated(uint32_t updateTime)
{

}
```

## IDataReaderSink 接口实现

### 获取基础数据管理器 get_basedata_mgr
```cpp
/**
 * @brief 获取基础数据管理器（IDataReaderSink接口实现）
 * 
 * 返回基础数据管理器的指针。
 * 
 * @return 返回基础数据管理器指针
 */
IBaseDataMgr* WtSimpDataMgr::get_basedata_mgr()
{
	return _runner->get_bd_mgr();
}
```

### 获取热点合约管理器 get_hot_mgr
```cpp
/**
 * @brief 获取热点合约管理器（IDataReaderSink接口实现）
 * 
 * 返回热点合约管理器的指针。
 * 
 * @return 返回热点合约管理器指针
 */
IHotMgr* WtSimpDataMgr::get_hot_mgr()
{
	return _runner->get_hot_mgr();
}
```

### 获取当前日期 get_date
```cpp
/**
 * @brief 获取当前日期（IDataReaderSink接口实现）
 * 
 * 返回当前日期。
 * 
 * @return 返回当前日期（格式：YYYYMMDD）
 */
uint32_t WtSimpDataMgr::get_date()
{
	return _cur_date;
}
```

### 获取当前分钟时间 get_min_time
```cpp
/**
 * @brief 获取当前分钟时间（IDataReaderSink接口实现）
 * 
 * 返回当前1分钟线时间。
 * 
 * @return 返回当前1分钟线时间（格式：HHMM）
 */
uint32_t WtSimpDataMgr::get_min_time()
{
	return _cur_min_time;
}
```

### 获取当前秒数 get_secs
```cpp
/**
 * @brief 获取当前秒数（IDataReaderSink接口实现）
 * 
 * 返回当前秒数（包括毫秒）。
 * 
 * @return 返回当前秒数（格式：SSmmm）
 */
uint32_t WtSimpDataMgr::get_secs()
{
	return _cur_secs;
}
```

### 数据读取器日志回调 reader_log
```cpp
/**
 * @brief 数据读取器日志回调（IDataReaderSink接口实现）
 * 
 * 当数据读取器需要记录日志时调用。
 * 
 * @param ll 日志级别
 * @param message 日志消息
 */
void WtSimpDataMgr::reader_log(WTSLogLevel ll, const char* message)
{
	WTSLogger::log_raw(ll, message);
}
```

## 私有初始化方法

### 初始化数据存储模块 initStore